In [ ]:
"""
==============================================================================
LANE DETECTION CON TRASFORMATA DI HOUGH
==============================================================================
Questo script rileva le linee di corsia (lane lines) in un'immagine stradale
usando OpenCV e la Trasformata di Hough Probabilistica.

INPUT:  strada2.jpg  (foto di autostrada, orientata in verticale)
OUTPUT: LaneDetection.jpg (immagine con corsia evidenziata in rosso,
        linee gialle di corsia e larghezza asfalto in pixel)

Autore: Generato con Claude (Anthropic)
==============================================================================
"""

# =============================================================================
# IMPORTAZIONE LIBRERIE
# =============================================================================
import cv2          # OpenCV: libreria per elaborazione immagini e computer vision
import numpy as np  # NumPy: libreria per calcoli numerici e matrici

# =============================================================================
# STEP 1: CARICAMENTO E ROTAZIONE DELL'IMMAGINE
# =============================================================================
# Carichiamo l'immagine originale dalla cartella
# cv2.imread() legge il file e lo trasforma in una matrice di pixel (altezza x larghezza x 3 canali colore)
img = cv2.imread('strada2.jpg')

# L'immagine originale è ruotata di 90° (il telefono era in verticale).
# La ruotiamo in senso antiorario per ottenere l'orientamento corretto (orizzontale).
# cv2.ROTATE_90_COUNTERCLOCKWISE = rotazione di 90° in senso antiorario
img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

# Salviamo le dimensioni dell'immagine per usarle dopo
# h = altezza (302 pixel), w = larghezza (535 pixel)
h, w = img.shape[:2]

# Creiamo una copia dell'immagine su cui disegneremo i risultati
# .copy() è importante: senza di esso, modificheremmo l'originale!
output = img.copy()


# =============================================================================
# STEP 2: CONVERSIONE IN SCALA DI GRIGI
# =============================================================================
# La Trasformata di Hough lavora su immagini in bianco e nero (1 solo canale).
# Convertiamo i 3 canali colore (Blu, Verde, Rosso) in un singolo canale di grigio.
# Formula usata da OpenCV: Grigio = 0.114*B + 0.587*G + 0.299*R
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


# =============================================================================
# STEP 3: SFOCATURA GAUSSIANA (RIDUZIONE DEL RUMORE)
# =============================================================================
# Prima di cercare i bordi, applichiamo una sfocatura per ridurre il "rumore"
# (piccole variazioni di luminosità che non sono bordi reali, come granelli, ombre, ecc.)
#
# Parametri:
#   (5, 5) = dimensione del kernel (finestra) di sfocatura.
#            Un kernel 5x5 pixel è un buon compromesso:
#            - Troppo piccolo (es. 3x3): non elimina abbastanza rumore
#            - Troppo grande (es. 11x11): sfoca anche i bordi veri delle strisce
#            - 5x5 è il valore standard per lane detection
#   0 = deviazione standard (calcolata automaticamente dal kernel)
blur = cv2.GaussianBlur(gray, (5, 5), 0)


# =============================================================================
# STEP 4: RILEVAMENTO BORDI CON CANNY
# =============================================================================
# L'algoritmo di Canny trova i "bordi" nell'immagine, ovvero i punti dove
# la luminosità cambia bruscamente (es. il confine tra asfalto grigio e striscia bianca).
#
# Parametri:
#   50  = soglia bassa (low threshold).
#         Pixel con gradiente < 50 vengono SCARTATI (non sono bordi).
#   150 = soglia alta (high threshold).
#         Pixel con gradiente > 150 sono SICURAMENTE bordi.
#         Pixel tra 50 e 150 sono bordi SOLO se collegati a un bordo sicuro.
#
# Come sono stati scelti 50 e 150:
#   - Il rapporto consigliato è 1:2 o 1:3 (qui è 1:3)
#   - Valori troppo bassi (es. 20, 60): troppi bordi falsi (rumore, ombre)
#   - Valori troppo alti (es. 100, 300): si perdono le strisce meno contrastate
#   - 50/150 è il range classico per strade ben illuminate
edges = cv2.Canny(blur, 50, 150)


# =============================================================================
# STEP 5: MASCHERA ROI (REGION OF INTEREST - REGIONE DI INTERESSE)
# =============================================================================
# Non ci interessa tutta l'immagine: il cielo, gli alberi e il guardrail
# non contengono linee di corsia utili. Consideriamo solo la METÀ INFERIORE
# dell'immagine, dove si trova la strada.
#
# Creiamo una "maschera" nera (tutti zeri) della stessa dimensione dei bordi
mask = np.zeros_like(edges)

# Definiamo un rettangolo che copre la metà inferiore dell'immagine:
#   (0, h)     = angolo in basso a sinistra
#   (0, h//2)  = angolo in alto a sinistra (metà altezza)
#   (w, h//2)  = angolo in alto a destra
#   (w, h)     = angolo in basso a destra
#
# h//2 = metà dell'altezza (151 pixel). Sopra questa linea, i bordi vengono ignorati.
# Perché metà immagine? Perché in una foto frontale di autostrada,
# la strada occupa circa la metà inferiore; sopra c'è il cielo e la vegetazione.
roi_vertices = np.array([[(0, h), (0, h // 2), (w, h // 2), (w, h)]], dtype=np.int32)

# Riempiamo di bianco (255) solo la zona ROI nella maschera
cv2.fillPoly(mask, roi_vertices, 255)

# Applichiamo la maschera: conserviamo i bordi SOLO nella zona ROI
# bitwise_and: ogni pixel viene mantenuto solo se entrambe le immagini hanno valore > 0
masked_edges = cv2.bitwise_and(edges, mask)


# =============================================================================
# STEP 6: TRASFORMATA DI HOUGH PROBABILISTICA
# =============================================================================
# La Trasformata di Hough rileva LINEE RETTE nei bordi trovati da Canny.
# Usiamo la versione "Probabilistica" (HoughLinesP) che è più veloce e
# restituisce direttamente i segmenti di linea (punto inizio e punto fine).
#
# Come funziona (in breve):
#   Ogni punto di bordo viene trasformato dallo spazio (x,y) allo spazio (rho, theta),
#   dove rho = distanza della linea dall'origine e theta = angolo della linea.
#   I punti che giacciono sulla stessa linea retta si "accumulano" nello stesso punto
#   (rho, theta). Cercando i picchi di accumulazione, troviamo le linee.
#
# Parametri:
#   rho = 1
#     Risoluzione della distanza in pixel. 1 pixel significa che cerchiamo linee
#     con precisione di 1 pixel. È il valore standard.
#
#   theta = np.pi/180
#     Risoluzione angolare in radianti. pi/180 = 1 grado di precisione.
#     È il valore standard che dà un buon compromesso velocità/precisione.
#
#   threshold = 50
#     Numero MINIMO di "voti" (punti di bordo allineati) perché un segmento
#     venga considerato una linea. Scelto come segue:
#     - Troppo basso (es. 20): troppe linee false (rumore, crepe nell'asfalto)
#     - Troppo alto (es. 100): si perdono le strisce più corte o sbiadite
#     - 50 è un buon valore per strisce di corsia ben visibili
#
#   minLineLength = 50
#     Lunghezza MINIMA del segmento in pixel. Segmenti più corti vengono ignorati.
#     - Troppo corto (es. 10): si rilevano piccoli bordi insignificanti
#     - Troppo lungo (es. 150): si perdono le strisce tratteggiate
#     - 50 pixel cattura le strisce tratteggiate senza troppo rumore
#
#   maxLineGap = 150
#     Distanza MASSIMA tra due segmenti per essere uniti in un'unica linea.
#     Questo è fondamentale per le strisce TRATTEGGIATE della corsia:
#     - Troppo piccolo (es. 10): ogni trattino è una linea separata
#     - Troppo grande (es. 300): linee diverse vengono unite erroneamente
#     - 150 pixel collega i trattini della stessa striscia senza confondere corsie diverse
lines = cv2.HoughLinesP(
    masked_edges,       # immagine dei bordi (in bianco e nero)
    rho=1,              # risoluzione distanza: 1 pixel
    theta=np.pi / 180,  # risoluzione angolo: 1 grado
    threshold=50,       # minimo 50 voti per confermare una linea
    minLineLength=50,   # lunghezza minima del segmento: 50 pixel
    maxLineGap=150      # gap massimo tra segmenti collegabili: 150 pixel
)


# =============================================================================
# STEP 7: CLASSIFICAZIONE LINEE (SINISTRA vs DESTRA)
# =============================================================================
# Le linee rilevate dalla Trasformata di Hough devono essere divise in due gruppi:
#   - Linee di SINISTRA: hanno pendenza NEGATIVA (scendono da sinistra a destra)
#   - Linee di DESTRA: hanno pendenza POSITIVA (salgono da sinistra a destra)
#
# Nota: in OpenCV l'asse Y è invertito (0 in alto, aumenta verso il basso),
# quindi le pendenze sono il contrario di quanto si vede visivamente.

# Liste per raccogliere pendenza e intercetta di ogni linea
left_params = []    # parametri linee di sinistra (pendenza, intercetta)
right_params = []   # parametri linee di destra (pendenza, intercetta)

for line in lines:
    # Ogni linea è definita da 4 coordinate: punto iniziale (x1,y1) e finale (x2,y2)
    x1, y1, x2, y2 = line[0]

    # Evitiamo la divisione per zero (linee perfettamente verticali)
    if x2 == x1:
        continue

    # Calcoliamo la pendenza (slope): quanto la linea sale/scende per ogni pixel orizzontale
    # Formula: pendenza = (variazione verticale) / (variazione orizzontale)
    slope = (y2 - y1) / (x2 - x1)

    # Calcoliamo l'intercetta: il valore di y quando x = 0
    # Formula: y = slope * x + intercept  →  intercept = y - slope * x
    intercept = y1 - slope * x1

    # Classifichiamo la linea in base alla pendenza:
    #   pendenza < -0.3 → linea di sinistra (la corsia va dal basso-sinistra in alto-destra)
    #   pendenza >  0.3 → linea di destra (la corsia va dal basso-destra in alto-sinistra)
    #   tra -0.3 e 0.3  → linea quasi orizzontale, la IGNORIAMO (bordi dell'immagine, guardrail, ecc.)
    #
    # Perché 0.3 come soglia?
    #   - Le linee di corsia hanno tipicamente pendenze tra 0.5 e 3.0
    #   - 0.3 filtra le linee quasi orizzontali (bordo inferiore, orizzonte)
    #   - Non è troppo restrittivo da perdere linee di corsia con angolo moderato
    if slope < -0.3:
        left_params.append((slope, intercept))
    elif slope > 0.3:
        right_params.append((slope, intercept))


# =============================================================================
# STEP 8: MEDIA DELLE LINEE (UNA LINEA PER LATO)
# =============================================================================
# Per ogni lato, facciamo la MEDIA di tutte le pendenze e intercette trovate.
# Questo produce una singola linea "rappresentativa" per ogni bordo della corsia,
# eliminando le piccole variazioni tra segmenti diversi della stessa striscia.

# Media dei parametri per il lato sinistro
left_slope = np.mean([s for s, b in left_params])
left_intercept = np.mean([b for s, b in left_params])

# Media dei parametri per il lato destro
right_slope = np.mean([s for s, b in right_params])
right_intercept = np.mean([b for s, b in right_params])


# =============================================================================
# STEP 9: CALCOLO DEI PUNTI DELLE LINEE (ESTESE OLTRE L'IMMAGINE)
# =============================================================================
# Calcoliamo dove ogni linea entra ed esce dall'immagine.
# Le linee vengono ESTESE oltre i bordi dell'immagine per un effetto più realistico
# (come si vede nell'immagine target LaneDetection.jpg).

def get_x(y, slope, intercept):
    """
    Data una coordinata y, calcola la x corrispondente sulla linea.
    Formula: y = slope * x + intercept  →  x = (y - intercept) / slope
    """
    return int((y - intercept) / slope)


# Estendiamo le linee oltre i bordi dell'immagine:
#   - In basso: 50 pixel sotto il bordo inferiore (y = h + 50)
#   - In alto: fino al bordo superiore (y = 0)
y_bottom_ext = h + 50   # sotto l'immagine (la linea "esce" dal basso)
y_top_ext = 0            # bordo superiore dell'immagine

# Punti della linea SINISTRA
left_x_bottom = get_x(y_bottom_ext, left_slope, left_intercept)
left_x_top = get_x(y_top_ext, left_slope, left_intercept)

# Punti della linea DESTRA
right_x_bottom = get_x(y_bottom_ext, right_slope, right_intercept)
right_x_top = get_x(y_top_ext, right_slope, right_intercept)


# =============================================================================
# STEP 10: AREA ROSSA DELLA CORSIA (OVERLAY SEMITRASPARENTE)
# =============================================================================
# Creiamo un poligono rosso semitrasparente che riempie l'area tra le due linee,
# evidenziando visivamente la corsia di marcia.

# Creiamo una copia per l'overlay (necessaria per la trasparenza)
overlay = output.copy()

# Definiamo i 4 vertici del poligono:
#   - Due punti in basso (dove le linee si trovano alla base dell'immagine)
#   - Due punti in alto (dove le linee sono a metà immagine, dove inizia la ROI)
y_poly_top = h // 2  # il poligono arriva fino a metà immagine (151 pixel)

poly_points = np.array([
    [get_x(h, left_slope, left_intercept), h],              # basso-sinistra
    [get_x(y_poly_top, left_slope, left_intercept), y_poly_top],  # alto-sinistra
    [get_x(y_poly_top, right_slope, right_intercept), y_poly_top], # alto-destra
    [get_x(h, right_slope, right_intercept), h]              # basso-destra
], dtype=np.int32)

# Riempiamo il poligono di ROSSO (in BGR: Blu=0, Verde=0, Rosso=255)
cv2.fillPoly(overlay, [poly_points], (0, 0, 255))

# Fondiamo l'overlay con l'immagine originale usando un'opacità (alpha) del 40%
# alpha = 0.4 → il rosso è al 40% e l'immagine originale al 60%
# Questo crea l'effetto "vetro colorato" che permette di vedere la strada sotto il rosso
alpha = 0.4
cv2.addWeighted(overlay, alpha, output, 1 - alpha, 0, output)


# =============================================================================
# STEP 11: DISEGNO DELLE LINEE GIALLE
# =============================================================================
# Disegniamo le due linee di corsia in GIALLO sopra l'overlay rosso.
# Giallo in BGR = (0, 255, 255) → Blu=0, Verde=255, Rosso=255
# Spessore = 2 pixel

# Linea sinistra (dal basso-sinistra verso l'alto)
cv2.line(output,
         (left_x_bottom, y_bottom_ext),    # punto di partenza (sotto l'immagine)
         (left_x_top, y_top_ext),          # punto di arrivo (bordo superiore)
         (0, 255, 255),                    # colore giallo (BGR)
         2)                                # spessore 2 pixel

# Linea destra (dal basso-destra verso l'alto)
cv2.line(output,
         (right_x_bottom, y_bottom_ext),   # punto di partenza (sotto l'immagine)
         (right_x_top, y_top_ext),         # punto di arrivo (bordo superiore)
         (0, 255, 255),                    # colore giallo (BGR)
         2)                                # spessore 2 pixel


# =============================================================================
# STEP 12: CALCOLO E VISUALIZZAZIONE LARGHEZZA ASFALTO
# =============================================================================
# Calcoliamo la distanza orizzontale tra le due linee di corsia ad una specifica
# altezza dell'immagine. Questa misura rappresenta la "larghezza dell'asfalto"
# in pixel in quel punto.
#
# La larghezza viene misurata approssimativamente a y = 94.3% dell'altezza immagine
# (circa 285 pixel), che corrisponde a una sezione della strada ben visibile
# nella parte bassa dell'inquadratura, appena sopra il bordo inferiore.
y_measure = int(h * 0.943)  # y ≈ 285 pixel

# Calcoliamo la posizione x delle due linee a questa altezza
x_left_at_measure = (y_measure - left_intercept) / left_slope
x_right_at_measure = (y_measure - right_intercept) / right_slope

# La larghezza è la differenza tra le due posizioni x
asphalt_width = x_right_at_measure - x_left_at_measure

# Scriviamo il testo sull'immagine
# Parametri di cv2.putText:
#   (10, 40)                    = posizione del testo (x=10, y=40 dall'angolo alto-sx)
#   cv2.FONT_HERSHEY_SIMPLEX   = tipo di font (semplice e leggibile)
#   0.9                         = scala del font (dimensione relativa)
#   (0, 0, 255)                 = colore ROSSO in BGR
#   2                           = spessore del testo in pixel
text = f"Asphalt width (px): {asphalt_width:.1f}"
cv2.putText(output, text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)


# =============================================================================
# STEP 13: SALVATAGGIO DELL'IMMAGINE RISULTANTE
# =============================================================================
# Salviamo l'immagine elaborata come file JPEG
cv2.imwrite('LaneDetection.jpg', output)

# Stampiamo un riepilogo dei risultati nella console
print("=" * 60)
print("LANE DETECTION COMPLETATA")
print("=" * 60)
print(f"Immagine di input:  strada2.jpg ({w}x{h} pixel)")
print(f"Immagine di output: LaneDetection.jpg")
print(f"")
print(f"Linee di corsia rilevate:")
print(f"  Linee a sinistra trovate: {len(left_params)}")
print(f"  Linee a destra trovate:   {len(right_params)}")
print(f"")
print(f"Parametri linea sinistra media:")
print(f"  Pendenza:   {left_slope:.3f}")
print(f"  Intercetta: {left_intercept:.1f}")
print(f"")
print(f"Parametri linea destra media:")
print(f"  Pendenza:   {right_slope:.3f}")
print(f"  Intercetta: {right_intercept:.1f}")
print(f"")
print(f"Larghezza asfalto a y={y_measure}: {asphalt_width:.1f} pixel")
print("=" * 60)

LANE DETECTION COMPLETATA
Immagine di input:  strada2.jpg (535x302 pixel)
Immagine di output: LaneDetection.jpg

Linee di corsia rilevate:
  Linee a sinistra trovate: 7
  Linee a destra trovate:   3

Parametri linea sinistra media:
  Pendenza:   -1.359
  Intercetta: 540.7

Parametri linea destra media:
  Pendenza:   0.873
  Intercetta: -118.6

Larghezza asfalto a y=284: 272.2 pixel
